In [ ]:
#code to read the data 

In [37]:
%pip install seaborn


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [41]:
#PAMPA2 data 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Define human-readable labels for the Activity IDs
activity_map = {
    0: 'Transient/Break', 1: 'Lying down', 2: 'Sitting', 3: 'Standing', 
    4: 'Walking', 5: 'Running', 6: 'Cycling', 7: 'Nordic walking', 
    9: 'Watching TV', 10: 'Computer work', 11: 'Car driving', 
    12: 'Ascending stairs', 13: 'Descending stairs', 16: 'Vacuum cleaning', 
    17: 'Ironing', 18: 'Folding laundry', 19: 'House cleaning', 
    20: 'Playing soccer', 24: 'Rope jumping'
}

# 2. build the 54 column headers based on dataset documentation
def get_pamap2_headers():
    headers = ['timestamp', 'activity_id', 'heart_rate']
    
    imu_features = [
        'temp', 
        'acc16_x', 'acc16_y', 'acc16_z', 
        'acc6_x', 'acc6_y', 'acc6_z', 
        'gyro_x', 'gyro_y', 'gyro_z', 
        'mag_x', 'mag_y', 'mag_z', 
        'orient_w', 'orient_x', 'orient_y', 'orient_z'
    ]
    
    for body_part in ['hand', 'chest', 'ankle']:
        for feature in imu_features:
            # Fixed the typo here!
            headers.append(f"{body_part}_{feature}")
            
    return headers

headers = get_pamap2_headers()
print(f"Total structured columns generated: {len(headers)}")
print("First 10 columns:", headers[:10])

Total structured columns generated: 54
First 10 columns: ['timestamp', 'activity_id', 'heart_rate', 'hand_temp', 'hand_acc16_x', 'hand_acc16_y', 'hand_acc16_z', 'hand_acc6_x', 'hand_acc6_y', 'hand_acc6_z']


# now we move to step 2: 

In [42]:
# Diagnostic Phase

# Ensure your 'subject101.dat' file is in the same folder as this notebook.
# (If you put it inside a folder called 'data', change this to 'data/subject101.dat')
#file_path = 'subject101.dat' 
file_path = 'pamap2+physical+activity+monitoring/PAMAP2_Dataset/Protocol/subject101.dat'

print("Loading raw subject file (this might take a few seconds because it is a massive telemetry file)...")
# We use sep='\s+' because the text file uses variable numbers of spaces to separate numbers
df_raw = pd.read_csv(file_path, sep='\s+', header=None)
df_raw.columns = headers

print(f"\n--- RAW DATA DIAGNOSTICS FOR SUBJECT 101 ---")
print(f"Total raw rows: {len(df_raw):,}")
print(f"Total continuous tracking time: {df_raw['timestamp'].max() / 60:.2f} minutes")

# 1. Check for missing data (NaNs) by percentage
missing_pct = df_raw.isnull().mean() * 100
print("\nTop 5 Columns with the Highest Percentage of Missing Data (NaNs):")
print(missing_pct.sort_values(ascending=False).head(5))

# 2. Check the distribution of activities this person did
print("\nRaw row counts per Activity ID:")
activity_counts = df_raw['activity_id'].value_counts()
for act_id, count in activity_counts.items():
    name = activity_map.get(act_id, 'Unknown Activity')
    print(f"  ID {act_id:<2} ({name}): {count:,} rows")

<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
/var/folders/_y/08rpnl7n2gs6yccr74gtrm640000gp/T/ipykernel_84591/1284695589.py:10: SyntaxWarning: invalid escape sequence '\s'
  df_raw = pd.read_csv(file_path, sep='\s+', header=None)


Loading raw subject file (this might take a few seconds because it is a massive telemetry file)...

--- RAW DATA DIAGNOSTICS FOR SUBJECT 101 ---
Total raw rows: 376,417
Total continuous tracking time: 62.88 minutes

Top 5 Columns with the Highest Percentage of Missing Data (NaNs):
heart_rate       90.864121
hand_orient_y     0.386274
hand_gyro_x       0.386274
hand_orient_z     0.386274
hand_orient_x     0.386274
dtype: float64

Raw row counts per Activity ID:
  ID 0  (Transient/Break): 126,460 rows
  ID 1  (Lying down): 27,187 rows
  ID 6  (Cycling): 23,575 rows
  ID 17 (Ironing): 23,573 rows
  ID 2  (Sitting): 23,480 rows
  ID 16 (Vacuum cleaning): 22,941 rows
  ID 4  (Walking): 22,253 rows
  ID 3  (Standing): 21,717 rows
  ID 5  (Running): 21,265 rows
  ID 7  (Nordic walking): 20,265 rows
  ID 12 (Ascending stairs): 15,890 rows
  ID 13 (Descending stairs): 14,899 rows
  ID 24 (Rope jumping): 12,912 rows
